# 02 — Position Panel

Turns the 7 raw quarterly info tables from notebook 01 into one tidy panel
and computes the concentration series (HHI, top-5 share, long/put/call
split) that motivates every later notebook.

In [1]:
import sys
from pathlib import Path
import re
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.utils import config as cfg
from src.edgar import parse
from src.risk import concentration

## 1. Re-parse each quarter's info table straight from the immutable raw files

In [2]:
rows = []
for period, accession in cfg.SA_13F_FILINGS:
    matches = list(cfg.EDGAR_13F.glob(f"{period}_{accession}_*"))
    info_file = next(f for f in matches if "primary_doc" not in f.name)
    positions = parse.parse_info_table(info_file.read_text())
    for p in positions:
        rows.append({
            "period": period, "issuer": p.issuer, "cusip": p.cusip,
            "value_usd": p.value_usd, "shares": p.shares, "put_call": p.put_call,
        })

panel = pd.DataFrame(rows)
panel.to_parquet(cfg.PROCESSED / "position_panel.parquet", index=False)
print(f"Panel: {len(panel)} rows across {panel.period.nunique()} quarters")
panel.head()

Panel: 152 rows across 7 quarters


,period,issuer,cusip,value_usd,shares,put_call
0,2024-12-31,CONSTELLATION ENERGY CORP,21037T109,21565644,96400,LONG
1,2024-12-31,MODINE MFG CO,607828100,7693926,66367,LONG
2,2024-12-31,MARVELL TECHNOLOGY INC,573874104,86802213,785896,LONG
3,2024-12-31,TALEN ENERGY CORP,87422Q109,27979751,138878,LONG
4,2024-12-31,VERTIV HOLDINGS CO,92537N108,51709137,455146,LONG


## 2. Concentration series — the headline numbers

In [3]:
conc = concentration.concentration_summary(panel)
conc["put_pct"] = conc.put_usd / conc.gross_usd
conc["long_pct"] = conc.long_usd / conc.gross_usd
conc["call_pct"] = conc.call_usd / conc.gross_usd
conc.to_csv(cfg.TABLES / "concentration_series.csv", index=False)
print(conc.to_string(index=False))

    period  n_positions   gross_usd    long_usd    put_usd   call_usd  top5_share      hhi  put_pct  long_pct  call_pct
2024-12-31            6   254813765   254813765          0          0    0.969806 0.231080 0.000000  1.000000  0.000000
2025-03-31           12  1005567727   545976373          0  459591354    0.760522 0.243262 0.000000  0.542953  0.457047
2025-06-30            9  2123023762  1099647394  570058608  453317760    0.817348 0.168025 0.268513  0.517963  0.213525
2025-09-30           28  4138368748  2242916747  887417600 1008034401    0.546060 0.082913 0.214437  0.541981  0.243583
2025-12-31           29  5516758345  3913647375    8910000 1594200970    0.600353 0.093524 0.001615  0.709411  0.288974
2026-03-31           42 13676657577  3855771552 8459056999 1361829026    0.486907 0.070425 0.618503  0.281924  0.099573
2026-06-30           26 20242292228 20169035068    5245000   68012160    0.772854 0.176297 0.000259  0.996381  0.003360


## 3. Cross-check against notebook 01's filed totals

Every quarter total already reconciled to its own `<tableValueTotal>` in
notebook 01 (see data/processed/reconciliation_log.md). Re-verify the panel
built here matches that log exactly, as an independent second check.

In [4]:
recon_log = (cfg.PROCESSED / "reconciliation_log.md").read_text()
recon_values = {}
for line in recon_log.splitlines():
    if not line.startswith("| 20"):
        continue
    cells = [c.strip() for c in line.strip("|").split("|")]
    period_str, filed_value_str = cells[0], cells[4]
    recon_values[period_str] = int(filed_value_str.replace("$", "").replace(",", ""))
mismatches = []
for _, row in conc.iterrows():
    filed_val = recon_values.get(row.period)
    if filed_val is None:
        continue
    if abs(row.gross_usd - filed_val) > 1:
        mismatches.append((row.period, row.gross_usd, filed_val))
print(f"Independent cross-check against notebook 01's log: {len(mismatches)} mismatches.")
assert not mismatches, f"Panel disagrees with reconciliation log: {mismatches}"

Independent cross-check against notebook 01's log: 0 mismatches.


## 4. Top holdings per quarter (for the data dictionary + README table)

In [5]:
top_by_q = {}
for period, g in panel.groupby("period"):
    total = g.value_usd.sum()
    top = g.assign(weight=g.value_usd / total).sort_values("weight", ascending=False).head(5)
    top_by_q[period] = top[["issuer", "put_call", "value_usd", "weight"]]
    print(f"\n{period} (top 5 of {len(g)}, total ${total:,.0f}):")
    for _, r in top.iterrows():
        print(f"  {r.weight:6.1%}  {r.put_call:5}  {r.issuer[:36]:36}  ${r.value_usd:,.0f}")


2024-12-31 (top 5 of 6, total $254,813,765):
   34.1%  LONG   MARVELL TECHNOLOGY INC                $86,802,213
   23.2%  LONG   VISTRA CORP                           $59,063,094
   20.3%  LONG   VERTIV HOLDINGS CO                    $51,709,137
   11.0%  LONG   TALEN ENERGY CORP                     $27,979,751
    8.5%  LONG   CONSTELLATION ENERGY CORP             $21,565,644

2025-03-31 (top 5 of 12, total $1,005,567,727):
   45.7%  CALL   Intel Corp                            $459,591,354
   11.7%  LONG   Broadcom Inc                          $117,201,000
    7.1%  LONG   Onto Innovation Inc                   $71,192,241
    6.1%  LONG   Vistra Corp                           $61,755,354
    5.5%  LONG   Modine Mfg Co                         $55,016,242

2025-06-30 (top 5 of 9, total $2,123,023,762):
   26.9%  PUT    VANECK ETF TRUST                      $570,058,608
   21.4%  CALL   INTEL CORP                            $453,317,760
   15.5%  LONG   BROADCOM INC                    

## 5. Data dictionary

In [6]:
data_dict = """# Data Dictionary — data/processed/position_panel.parquet

| Field | Type | Description |
|---|---|---|
| period | str (YYYY-MM-DD) | 13F-HR reporting quarter-end date |
| issuer | str | Issuer name as filed (not normalised across quarters — e.g. "SanDisk Corp" spelling is as-filed) |
| cusip | str | 9-character CUSIP identifier, as filed |
| value_usd | int | Section 13(f) fair market value, whole US dollars (post-2023 SEC amendment) |
| shares | int | Number of shares or principal amount |
| put_call | str | "LONG" (default/blank), "PUT", or "CALL" — options entries give value of the UNDERLYING only; no strike, expiry, or bought/written flag is disclosed |

## Derived fields (outputs/tables/concentration_series.csv)
| Field | Description |
|---|---|
| gross_usd | Sum of value_usd across all entries that quarter |
| top5_share | Share of gross_usd held in the 5 largest positions |
| hhi | Herfindahl-Hirschman index of position weights (sum of squared weights) |
| put_pct / long_pct / call_pct | Share of gross_usd in each instrument type |

## Known data-quality notes
- 2025-12-31 filing has a $1 rounding gap between its own summary total and
  the sum of its line items (see reconciliation_log.md) — immaterial, not
  corrected, documented.
- Two related CIKs (2045724, 2038540) file identical holdings for Q1 2026;
  only CIK 2045724 (Situational Awareness LP) is used throughout this panel.
"""
(cfg.REPORTS / "DATA_DICTIONARY.md").write_text(data_dict)
print("Data dictionary written to reports/DATA_DICTIONARY.md")
print("\nNotebook 02 complete.")

Data dictionary written to reports/DATA_DICTIONARY.md

Notebook 02 complete.
